# Flanker Analysis in Nilearn — Part 2: Second-Level (Within-Subject, Across-Run) Analysis

In FSL's terminology, a **second-level analysis** combines the two runs of each subject into a single subject-level estimate using a **fixed-effects** model. Nothing here generalizes to a population — it is only about getting the best estimate *for each individual*.

**Before you start:** you need to have run **Part 1**. This notebook reads the effect-size and variance maps that Part 1 saved to your output folder.

**What you will do:**
1. For each subject and each contrast, combine the 2 run-level maps with `compute_fixed_effects`.
2. Compare the run-level (Part 1) maps with the subject-level (this notebook) maps.

**What to submit:** answers to the 3 reflection questions at the end, in a separate Word or text file, uploaded to Canvas.

## Step 0 — Tell the notebook where your data are
Use the **same three paths** as in Part 1.

In [ ]:
# ---- EDIT THESE THREE PATHS, THEN RUN THE REST OF THE NOTEBOOK UNCHANGED ----
BIDS_DIR     = "/Users/mexico/Documents/Teaching/fundamentals_human_brain_imaging/datasets/flanker/rawdata"                       # contains sub-01/func/*_events.tsv
FMRIPREP_DIR = "/Users/mexico/Documents/Teaching/fundamentals_human_brain_imaging/datasets/flanker/derivatives/fmriprep_outputs/"  # contains sub-01/func/*_desc-preproc_bold.nii.gz
OUTPUT_DIR   = "/Users/mexico/Documents/Teaching/fundamentals_human_brain_imaging/datasets/flanker/derivatives/nilearn_outputs"   # where YOUR results will be saved

## Step 1 — Imports and settings (largely mirrors Level1 notebook)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import nilearn

BIDS_DIR, FMRIPREP_DIR, OUTPUT_DIR = Path(BIDS_DIR), Path(FMRIPREP_DIR), Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Fixed settings for this assignment (you do not need to change these) ----
SUBS = ["sub-01", "sub-02", "sub-03"]
RUNS = ["run-1", "run-2"]
SPACE = "space-MNI152NLin2009cAsym_res-2"

# Same contrasts dictionary as defined in the level1 notebook
CONTRASTS = {
    "incongruent":           {"incongruent": 1.0},
    "congruent":             {"congruent": 1.0},
    "incongruent-congruent": {"incongruent": 1.0, "congruent": -1.0},
    "all-baseline":          {"incongruent": 0.5, "congruent": 0.5},
}

# ---- Where files live (layout inside each folder follows BIDS / fMRIPrep) ----
def events_path(sub, run):
    return BIDS_DIR / sub / "func" / f"{sub}_task-flanker_{run}_events.tsv"

def fmriprep_path(sub, run, suffix):
    return FMRIPREP_DIR / sub / "func" / f"{sub}_task-flanker_{run}_{suffix}"

def bold_path(sub, run):  return fmriprep_path(sub, run, f"{SPACE}_desc-preproc_bold.nii.gz")
def mask_path(sub, run):  return fmriprep_path(sub, run, f"{SPACE}_desc-brain_mask.nii.gz")
def confounds_path(sub, run): return fmriprep_path(sub, run, "desc-confounds_timeseries.tsv")

# ---- Where this assignment writes its outputs ----
STAT_KEYS = {"effect": "effect_size", "variance": "effect_variance", "z": "z_score"}

def level1_path(sub, run, contrast, stat):
    return OUTPUT_DIR / "level1" / sub / f"{sub}_{run}_contrast-{contrast}_{stat}.nii.gz"

def level2_path(sub, contrast, stat):
    return OUTPUT_DIR / "level2" / sub / f"{sub}_contrast-{contrast}_{stat}.nii.gz"

def level3_path(contrast, smoothing, stat):
    return OUTPUT_DIR / "level3" / f"group_contrast-{contrast}_smooth-{smoothing}_{stat}.nii.gz"


from nilearn.glm.contrasts import compute_fixed_effects
from nilearn.masking import intersect_masks, apply_mask
from nilearn.plotting import plot_stat_map

In [ ]:
# Check that Part 1 finished: every run-level effect, variance, and z-map should exist
missing = [level1_path(s, r, c, st) for s in SUBS for r in RUNS for c in CONTRASTS for st in STAT_KEYS
           if not level1_path(s, r, c, st).exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} Part 1 outputs are missing. Run Part 1 first. Example: {missing[0]}")
missing_mask = [mask_path(s, r) for s in SUBS for r in RUNS if not mask_path(s, r).exists()]
if missing_mask:
    raise FileNotFoundError(f"fMRIPrep masks not found, check FMRIPREP_DIR. Example: {missing_mask[0]}")
print("Part 1 outputs found.")

### How does this differ from the FSL analysis? (conceptual)

| Step | FSL / FEAT | Nilearn (this notebook) |
|---|---|---|
| Inputs | `cope` and `varcope` images from each run | `effect_size` and `effect_variance` images from each run |
| Model | Higher-level GLM with one regressor per subject; "fixed effects" option | `compute_fixed_effects` combines the run-level maps directly |
| Run weighting | Runs are weighted by their variance | By default an **unweighted** average of the run effects (inverse-variance weighting is available with `precision_weighted=True`) |
| Statistic | `zstat` from FEAT | We divide the combined effect by its standard error (effect ÷ √variance) |

In both cases, **fixed effects** means we treat the runs as the only things being measured: the result applies to *this subject's* data, not to people in general.

## Step 2 — Combine the runs within each subject

For each subject and each contrast, `compute_fixed_effects` takes the two run-level **effect** maps and the two run-level **variance** maps and returns a combined effect map and a combined variance map. Because the combined variance is usually smaller than either run's variance, dividing the effect by its standard error gives larger statistics than either run alone, if the effect is real and consistent.

We restrict the calculation to voxels that are inside the brain mask in **both** runs (the *intersection* of the two fMRIPrep masks).

In [ ]:
SUBJECT_MASKS = {}

for sub in SUBS:
    # Brain mask = voxels that are inside the mask in BOTH runs
    mask = intersect_masks([str(mask_path(sub, run)) for run in RUNS], threshold=1)
    SUBJECT_MASKS[sub] = mask
    level2_path(sub, "incongruent", "effect").parent.mkdir(parents=True, exist_ok=True)

    for cname in CONTRASTS:
        effects   = [nib.load(str(level1_path(sub, run, cname, "effect")))   for run in RUNS]
        variances = [nib.load(str(level1_path(sub, run, cname, "variance"))) for run in RUNS]

        outputs = compute_fixed_effects(effects, variances, mask=mask)
        fx_effect, fx_variance = outputs[0], outputs[1]  

        eff = fx_effect.get_fdata()
        var = fx_variance.get_fdata()
        with np.errstate(divide="ignore", invalid="ignore"):
            z = np.where(var > 0, eff / np.sqrt(var), 0.0)   # effect / standard error to get z-statistic
        fx_z = nib.Nifti1Image(z.astype("float32"), fx_effect.affine)

        fx_effect.to_filename(str(level2_path(sub, cname, "effect")))
        fx_variance.to_filename(str(level2_path(sub, cname, "variance")))
        fx_z.to_filename(str(level2_path(sub, cname, "z")))
    print(f"{sub}: saved fixed-effects maps for {len(CONTRASTS)} contrasts")

## Step 3 — Compare run-level and subject-level maps

Below, for each subject, are three maps of the **incongruent − congruent** z-statistic: run 1, run 2, and the fixed-effects combination of both. All three use the same color scale (and, as before, no statistical threshold). 

Look for: Do the two runs agree with each other? Does the combined map look cleaner or stronger than the individual runs?

In [ ]:
CONTRAST_TO_SHOW = "incongruent-congruent"

fig, axes = plt.subplots(len(SUBS), 3, figsize=(16, 3.2 * len(SUBS)))
for i, sub in enumerate(SUBS):
    panels = [(f"{sub} run-1", level1_path(sub, "run-1", CONTRAST_TO_SHOW, "z")),
              (f"{sub} run-2", level1_path(sub, "run-2", CONTRAST_TO_SHOW, "z")),
              (f"{sub} both runs (fixed effects)", level2_path(sub, CONTRAST_TO_SHOW, "z"))]
    for ax, (title, path) in zip(axes[i], panels):
        plot_stat_map(str(path), axes=ax, display_mode="z", cut_coords=[20, 35, 50], vmax=5, title=title)
plt.show()

### Quantifying the comparison

Two simple numbers for each subject and contrast:
- **Run-to-run correlation:** the correlation, across all brain voxels, between the run-1 and run-2 z-maps. A higher value means the two runs tell a more similar story.
- **Mean |z|:** the average absolute z-value across brain voxels, for run 1, run 2, and the combined map. 

(A mean |z| of about 0.8 is what you would expect from pure noise, so values noticeably above that suggest real signal.)

In [ ]:
rows = []
for sub in SUBS:
    for cname in CONTRASTS:
        z1 = apply_mask(str(level1_path(sub, "run-1", cname, "z")), SUBJECT_MASKS[sub])
        z2 = apply_mask(str(level1_path(sub, "run-2", cname, "z")), SUBJECT_MASKS[sub])
        zf = apply_mask(str(level2_path(sub, cname, "z")), SUBJECT_MASKS[sub])
        rows.append({"subject": sub, "contrast": cname,
                     "run1-run2 correlation": np.corrcoef(z1, z2)[0, 1],
                     "mean |z| run-1": np.abs(z1).mean(),
                     "mean |z| run-2": np.abs(z2).mean(),
                     "mean |z| combined": np.abs(zf).mean()})
display(pd.DataFrame(rows).round(2).set_index(["subject", "contrast"]))

## Reflection questions

Write your answers in a **separate Word document or text file** (not in this notebook) and upload it to Canvas. A few sentences for each is enough.

1. **What fixed effects does and doesn't tell you.** Explain what a fixed-effects analysis across runs assumes, and why its results can be interpreted only for the individual subject and not for the population the subject came from. Which source of variability is being ignored?

2. **Level 1 vs. level 2.** Compare the run-level and combined maps and the table above. What changed when the two runs were combined, and why? What does the run-to-run correlation tell you about how reliable a single run is?

3. **Why variance maps?** Part 1 saved a *variance* map in addition to the effect map for every contrast. What does `compute_fixed_effects` use the variance for? Describe a situation in which one run is much noisier than the other, and explain why weighting the runs by their precision (`precision_weighted=True`) would give a different answer than a simple average.